In [1]:
print("hello")

hello


## 정답지 (AI)

In [ ]:
# 네이버 블로그 크롤러

# Step 1. 필요한 모듈 로딩
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time
import math
import pandas as pd

# Step 2. 사용자 입력
print("=" * 80)
print("  이 크롤러는 네이버 블로그 수집용 웹크롤러입니다.")
print("=" * 80)
query_txt = input('1. 수집할 키워드를 입력하세요 (예: 서진수 빅데이터): ')
fc_name   = input('2. 저장할 CSV 파일 경로 (예: c:\\py_temp\\blog.csv): ')
fx_name   = input('3. 저장할 XLSX 파일 경로 (예: c:\\py_temp\\blog.xlsx): ')

# Step 3. 크롬 드라이버 실행
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)
driver.get('https://www.naver.com/')
driver.maximize_window()
time.sleep(2)

# Step 4. 검색어 입력 후 검색
element = driver.find_element(By.ID, 'query')
element.click()
element.send_keys(query_txt)
element.send_keys(Keys.ENTER)
time.sleep(2)

# Step 5. 블로그 탭 클릭
driver.find_element(By.LINK_TEXT, '블로그').click()
time.sleep(2)

# Step 6. 수집 건수 입력
collect_cnt = int(input('몇 건을 수집하시겠습니까? (예: 15): '))
collect_page_cnt = math.ceil(collect_cnt / 10)
print(f'{collect_cnt}건 수집을 위해 {collect_page_cnt} 페이지를 조회합니다.')
print('=' * 80)

# Step 1~6 동일 (위 코드 그대로 사용)

# Step 7. 데이터 수집 (완전히 새로운 방식)
no2 = []; title2 = []; cont2 = []; date2 = []; bname2 = []
no = 1
seen_titles = set()

for a in range(1, collect_page_cnt + 1):
    time.sleep(2)
    html = driver.page_source
    soup = BeautifulSoup(html, 'html.parser')

    # ✅ 랜덤 클래스 대신 절대 안 바뀌는 data-template-id 속성 사용
    posts = soup.find_all('div', attrs={'data-template-id': 'ugcItem'})

    for post in posts:

        # 제목: sds-comps-text-type-headline1 포함된 span
        try:
            title = post.find('span', class_='sds-comps-text-type-headline1').get_text()
        except:
            continue

        if title in seen_titles:
            continue
        seen_titles.add(title)

        # 요약 내용: fds-ugc-ellipsis 포함된 a 태그
        try:
            cont_tag = post.find('a', class_='fds-ugc-ellipsis3') or \
                       post.find('a', class_='fds-ugc-ellipsis2')
            cont = cont_tag.get_text() if cont_tag else ''
        except:
            cont = ''

        # ✅ 날짜: data-sds-comp="Profile" 안의 subtext span
        try:
            profile = post.find('div', attrs={'data-sds-comp': 'Profile'})
            date = profile.find('span', class_='sds-comps-profile-info-subtext').get_text()
        except:
            date = ''

        # ✅ 닉네임: Profile 안의 title-text span > a > span
        try:
            profile = post.find('div', attrs={'data-sds-comp': 'Profile'})
            b_name = profile.find('span', class_='sds-comps-profile-info-title-text').get_text()
        except:
            b_name = ''

        print(f'1. 번호: {no}')
        print(f'2. 제목: {title}')
        print(f'3. 요약내용: {cont}')
        print(f'4. 작성일자: {date}')
        print(f'5. 블로그 닉네임: {b_name}')
        print('-' * 60)

        no2.append(no); title2.append(title); cont2.append(cont)
        date2.append(date); bname2.append(b_name)

        no += 1
        if no > collect_cnt:
            break

    if no > collect_cnt:
        break

    time.sleep(1)
    try:
        driver.find_element(By.LINK_TEXT, str(a + 1)).click()
    except:
        try:
            driver.find_element(By.LINK_TEXT, '다음 페이지로').click()
        except:
            print('더 이상 페이지가 없습니다.')
            break

print("요청하신 작업이 모두 완료되었습니다")

# Step 8. 파일 저장
df = pd.DataFrame({
    '번호': no2, '제목': title2, '요약내용': cont2,
    '작성일자': date2, '블로그닉네임': bname2
})
df.to_excel(fx_name, index=False, engine='openpyxl')
df.to_csv(fc_name, index=False, encoding='utf-8-sig')
print('저장 완료!')
print(df)

## 배운 내용 안에서

## Chap 1~7 

In [ ]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 NaverBlog 사이트의 논문 및 학술자료 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')

#Step 3. 수집된 데이터를 저장할 파일 이름 입력받기
ft_name = input('2.결과를 저장할 txt형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.txt): ')
fc_name = input('3.결과를 저장할 csv형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.csv): ')
fx_name = input('4.결과를 저장할 xlsx형식의 파일명을 쓰세요(예: c:\\py_temp\\riss.xlsx): ')

#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.riss.kr/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'학위논문').click()
time.sleep(2)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find('div','srchResultListW').find_all('li')
for i in content_1 :
    print(i.get_text().replace("\n",""))

## Chap 8

In [ ]:
#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
total_cnt = soup_1.find('div','searchBox pd').find('span','num').get_text() #총 몇건인지 알려줌
print('키워드 %s (으)로 총 %s 건의 학위논문이 검색되었습니다' %(query_txt,total_cnt)) 
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
collect_page_cnt = math.ceil(collect_cnt / 10) #최대 페이지 찾기 (한페이지에 10건이기에 수정사항임)
print('%s 건을 수집하기 위해 %s 페이지를 조회합니다.' %(collect_cnt,collect_page_cnt))
print('=' *80)

## Chap 9

In [ ]:
# ********여기부터 제일 중요함

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #논문제목 저장
writer2 = [ ] #논문저자 저장
org2 = [ ] #소속기관 저장
no = 1		# 논문 번호 쓰려고 넘버해놓음

# **반복문 두개이기에 들여쓰기 잘해야 함

#여기부터 반복문 2개 (왜? 15건 뽑는다고 하면, 1페이지 일때 1~10건, 2페이지일 때 1~5건)
for a in range(1, collect_page_cnt + 1) :		# 페이지

    html_2 = driver.page_source 		#1페이지 전체 소스 다 가져오는 코드 / 들여쓰기 주의
    soup_2 = BeautifulSoup(html_2, 'html.parser')

    content_2 = soup_2.find('div','srchResultListW').find_all('li')	 #1페이지 일 때 10건 다 가져오라는 코드

    for b in content_2 :		# 첫번 째 li 태그 빼옴
        #1. 논문제목 있을 경우만
        try :
            title = b.find('div','cont').find('p','title').get_text() 	#li가 여러개일 때 제목 있는 애들은 이렇게 뽑음
        except :
            continue
        else :
            f = open(ft_name, 'a' , encoding="UTF-8")		#open : 준비 / a 쓰면 추가, w 쓰면 덮어쓰기(w쓰면 100건 크롤링해도 1건만 남음)
            print('1.번호:',no)
            no2.append(no)
            f.write('\n'+'1.번호:' + str(no))		# write : 쓰기 / 글자, 숫자면 안 맞아서 에러남. 그래서 str 사용하여 글자로 바꿈

            print('2.논문제목:',title)		# 74번줄에서 데이터 뽑음
            title2.append(title)		# 텍스트파일에 제목추가
            f.write('\n' + '2.논문제목:' + title)	

            writer = b.find('span','writer').get_text()		#논문저자 찾기
            print('3.저자:',writer)
            writer2.append(writer)		# 텍스트파일에 저자추가
            f.write('\n' + '3.저자:' + writer)

            org = b.find('span','assigned').get_text()		#소속기관 찾기
            print('4.소속기관:' , org)
            org2.append(org)		# 기관 추가
            f.write('\n' + '4.소속기관:' + org + '\n')

            f.close( )			# close : 파일 저장
            if no > collect_cnt :		# 사용자가 요청한 건수
                break		# 요청대로 되면 멈춤

            no += 1
            print("\n")

    time.sleep(1) # 페이지 변경 전 1초 대기

    a += 1		# 여기까지는 숫자이기에 아래서 글자로 바꿈. 글자만 누를 수 있기에
    b = str(a)		#여기서 숫자로

    try :
        driver.find_element(By.LINK_TEXT ,'%s' %b).click()
    except :
        driver.find_element(By.LINK_TEXT('다음 페이지로')).click()
        print("요청하신 작업이 모두 완료되었습니다")


# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd		#117~123까지 표 만드는 기능 (pandas 사용) 

# 표 만들어줘. 참고로 데이터프레임은 **빈칸** 있으면 안됨. 0으로 채우거나 아무 글자 집어넣어야함.
# 다만! pd.DataFrame() 이건 빈칸 있어도 그대로 해달라는 뜻임. 
df = pd.DataFrame()		
df['번호']=no2
df['제목']=pd.Series(title2)
df['저자']=pd.Series(writer2)
df['소속(발행)기관']=pd.Series(org2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, encoding="utf-8" , engine='openpyxl')		#표이름. 표이름을 엑셀로 저장해달라는 뜻
#engine='openpyxl' 이건 안 써도 됨. 웬만하면 자동으로 되어있음.

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

## 새롭게 시작

In [1]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 NAVER 블로그 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')


#Step 3. 수집된 데이터를 저장할 파일 이름 및 경로 자동 설정하기
import os
from datetime import datetime

# 1. 저장할 기본 폴더 경로 설정 (경로 마지막에 꼭 \\ 를 붙여주세요)
save_path = "C:\\py_temp\\추출 후 파일 저장\\"

# 해당 폴더가 컴퓨터에 없으면 자동으로 생성해 주는 안전장치
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2. '검색어_현재시간' 형태로 파일명 자동 생성
now = datetime.now().strftime("%Y%m%d_%H%M") # 파일명 중복을 막기 위한 현재 시간(년월일_시분)

fc_name = f"{save_path}{query_txt}_{now}.csv"
fx_name = f"{save_path}{query_txt}_{now}.xlsx"

print(f"👉 수집된 데이터는 [{save_path}] 폴더에 자동 저장됩니다.")


#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.naver.com/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")
time.sleep(3)

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'블로그').click()
time.sleep(3)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find_all('div','sds-comps-vertical-layout')
for i in content_1 :
    print(i.get_text().replace("\n",""))





#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
print('%s 건을 수집하기 위해 페이지를 조회합니다.' %(collect_cnt))
print('=' *80)

# ********여기부터 제일 중요함

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #블로그제목 저장
cont2 = [ ] #블로그내용 저장
date2 = [ ] #작성일자 저장
name2 = [ ] #블로그닉네임 저장
no = 1		# 논문 번호 쓰려고 넘버해놓음

time.sleep(3)

html_2 = driver.page_source 		#1페이지 전체 소스 다 가져오는 코드 / 들여쓰기 주의
soup_2 = BeautifulSoup(html_2, 'html.parser')

blog_list_2 = soup_2.find_all('ul','sds-comps-vertical-layout sds-comps-full-layout Rpk3YFQcZBMzoLEWz9U_')

for i in blog_list_2 :		# 첫번 째 li 태그 빼옴
    #1. 논문제목 있을 경우만
    try :
        title = i.find('span','sds-comps-text sds-comps-text-ellipsis sds-comps-text-ellipsis-1 sds-comps-text-type-headline1 sds-comps-text-weight-sm').get_text()   	#li가 여러개일 때 제목 있는 애들은 이렇게 뽑음
    except :
        continue
    else :
        print('1.번호:',no)
        no2.append(no)

        print('2.블로그제목:',title)		# 74번줄에서 데이터 뽑음
        title2.append(title)		# 텍스트파일에 제목추가

        cont = i.find('span','sds-comps-text sds-comps-text-type-body1 sds-comps-text-weight-sm').get_text()		#내용 찾기
        print('3.내용:',cont)
        cont2.append(cont)		# 텍스트파일에 저자추가

        datename = i.find('div','sds-comps-horizontal-layout').find_all('span','sds-comps-text').get_text()		#작성일자 찾기
        print('4.작성일자:' , date)
        date2.append(datename)		# 기관 추가
        name2.append(datename)

        print("=" * 50) # 보기 편하게 구분선 추가

        if no > collect_cnt :		# 사용자가 요청한 건수
            break		# 요청대로 되면 멈춤

        no += 1
        print("\n")

time.sleep(1) # 페이지 변경 전 1초 대기
print("요청하신 작업이 모두 완료되었습니다")


# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd		#117~123까지 표 만드는 기능 (pandas 사용) 

# 표 만들어줘. 참고로 데이터프레임은 **빈칸** 있으면 안됨. 0으로 채우거나 아무 글자 집어넣어야함.
# 다만! pd.DataFrame() 이건 빈칸 있어도 그대로 해달라는 뜻임. 
df = pd.DataFrame()		
df['번호']=no2
df['제목']=pd.Series(title2)
df['내용']=pd.Series(cont2)
df['작성일자']=pd.Series(date2)
df['블로그닉네임']=pd.Series(name2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, engine='openpyxl')		#표이름. 표이름을 엑셀로 저장해달라는 뜻
#engine='openpyxl' 이건 안 써도 됨. 웬만하면 자동으로 되어있음.

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 NAVER 블로그 수집용 웹크롤러입니다.


👉 수집된 데이터는 [C:\py_temp\추출 후 파일 저장\] 폴더에 자동 저장됩니다.
강사섭외는 스타강사랩2025.07.16.Keep에 저장Keep에 바로가기빅데이터 전문가 서진수 강사 '4차산업혁명, 무엇을 대비해야하나' 특강기업체나 공공기관 등 강연이 필요한 곳이라면 이제 스타강사랩과 함께해 보세요 인문학 역사학 동기부여 성공학 행복특강 등 수많은 분야의 전문가들을 맞춤섭외해 드립니다 평창농협 주최로 4차산업혁명 특강이 진행되었어요 강연자로는 빅데이터 전문가 서진수 강사가 함께해 주었는데요 '4차산업혁명, 무엇을 대비해야하나'라는 주제로 열띤 강연을 펼쳐주었어요... 뉴턴의 과학실 과학전문학원2022.08.06.Keep에 저장Keep에 바로가기전문가에게 길을 묻다. 빅데이터 전문가편 서진수 대표빅데이터 전문가편을 진행하였습니다. 각종 티비 프로그램과 인공지능과 관련된 18권의 책 출판, 국회에 4차 산업혁명 특별 자문위원, 세계인명사전에 과학자로 등재된 서진수 대표님과 함께 했습니다. 예전에 강연을 잘하셔서 국회의원 상을 받았다고 들었는데 역시나 아이들 눈맞춤으로 강연을 잘해주셨어요. 한시간 조금 넘게 진행하였는데, 아이들 모두... 6한국정보인재개발원-빅데이터,파이썬,R활용,6시그마2025.09.09.Keep에 저장Keep에 바로가기[한국정보인재개발원_충북대학교] 줌(Zoom)을 이용한 비대면 교육 - R프로그래밍 활용 빅데이터 교육 후기(2025.08.04~07)R프로그램은 오픈소스 프로그램으로 통계/데이터 마이닝 및 그래프를 위한 언어입니다. 최근 빅데이터 분석을 목적으로 기업에서 많이 쓰이고 있으며 오픈 소스 프로그램이기 때문에 무료로 사용 가능하단 점이 큰 장점입니다. 이번 R프로그래밍 교육은 서진수 강사님께서 진행해주셨습니다.^^ 요즘 컴퓨터 언어에 대한 관심이 높아지고 있습니다! R프로그램을 활용한... 6[강원대학교] 빅데이터 분석 활용 교육 후기 (2022.08.01~05)[아주대학교] 줌(Zoom)을 이용한 비대면 교육 - 빅데이터분석

## 연습용

In [64]:
# riss.kr 에서 특정 키워드로 논문 / 학술 자료 검색하기
#Step 1. 필요한 모듈을 로딩합니다
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
import time

#Step 2. 사용자에게 검색 관련 정보들을 입력 받습니다.
print("=" *100)
print(" 이 크롤러는 NAVER 블로그 수집용 웹크롤러입니다.")
print("=" *100)
query_txt = input('1.수집할 자료의 키워드는 무엇입니까? : ')


#Step 3. 수집된 데이터를 저장할 파일 이름 및 경로 자동 설정하기
import os
from datetime import datetime

# 1. 저장할 기본 폴더 경로 설정 (경로 마지막에 꼭 \\ 를 붙여주세요)
save_path = "C:\\py_temp\\추출 후 파일 저장\\"

# 해당 폴더가 컴퓨터에 없으면 자동으로 생성해 주는 안전장치
if not os.path.exists(save_path):
    os.makedirs(save_path)

# 2. '검색어_현재시간' 형태로 파일명 자동 생성
now = datetime.now().strftime("%Y%m%d_%H%M") # 파일명 중복을 막기 위한 현재 시간(년월일_시분)

fc_name = f"{save_path}{query_txt}_{now}.csv"
fx_name = f"{save_path}{query_txt}_{now}.xlsx"

print(f"👉 수집된 데이터는 [{save_path}] 폴더에 자동 저장됩니다.")


#Step 4. 크롬 드라이버 설정 및 웹 페이지 열기
s = Service("c:/py_temp/chromedriver.exe")
driver = webdriver.Chrome(service=s)

url = 'https://www.naver.com/'
driver.get(url)
time.sleep(5)
driver.maximize_window()

#Step 5. 자동으로 검색어 입력 후 조회하기
element = driver.find_element(By.ID,'query')
driver.find_element(By.ID,'query').click( )
element.send_keys(query_txt)
element.send_keys("\n")
time.sleep(3)

#Step 6.학위 논문 선택하기
driver.find_element(By.LINK_TEXT,'블로그').click()
time.sleep(3)

#Step 7.Beautiful Soup 로 본문 내용만 추출하기
from bs4 import BeautifulSoup
html_1 = driver.page_source
soup_1 = BeautifulSoup(html_1, 'html.parser')

content_1 = soup_1.find_all('div','spw_fsolid  type_head _fsolid_head  _slog_visible')
for i in content_1 :
    print(i.get_text().replace("\n",""))





#Step 8. 총 검색 건수를 보여주고 수집할 건수 입력받기
import math
collect_cnt = int(input('이 중에서 몇 건을 수집하시겠습니까?: '))
print('%s 건을 수집하기 위해 페이지를 조회합니다.' %(collect_cnt))
print('=' *80)

# ********여기부터 제일 중요함

#Step 9. 각 항목별로 데이터를 추출하여 리스트에 저장하기
no2 = [ ] #번호 저장
title2 = [ ] #블로그제목 저장
cont2 = [ ] #블로그내용 저장
date2 = [ ] #작성일자 저장
name2 = [ ] #블로그닉네임 저장
no = 1      # 논문 번호 쓰려고 넘버해놓음

time.sleep(3)

html_2 = driver.page_source         
soup_2 = BeautifulSoup(html_2, 'html.parser')

# 1. 네이버의 무작위 클래스를 뚫는 '부분 일치(lambda)' 방식 적용
# 네이버가 주로 쓰는 블로그 글 바구니(컨테이너) 클래스를 유연하게 모두 잡습니다.
content_2 = soup_2.find_all(class_=lambda x: x and ('api_ani_send' in x or 'user_box_inner' in x or 'detail_box' in x))

# 만약 위 방법으로도 못 잡으면, 전통적인 bx 클래스로 다시 시도
if not content_2:
    content_2 = soup_2.find_all('li', class_='bx')

for b in content_2 :        
    # 2. 제목 찾기 ('api_subject_bx' 라는 단어가 포함된 이름표는 무조건 통과)
    try :
        title_tag = b.find(class_=lambda x: x and 'api_subject_bx' in x)
        if not title_tag: # 제목 태그가 없으면 엉뚱한 광고 블록이므로 건너뜀
            continue
        title = title_tag.get_text().strip()       
    except :
        continue 

    print('1.번호:', no)
    no2.append(no)

    print('2.블로그제목:', title)        
    title2.append(title)        

    # 3. 내용 찾기 ('txt_lines' 또는 사용자님이 쓰셨던 'full-layout' 단어 추적)
    try:
        cont_tag = b.find(class_=lambda x: x and ('api_txt_lines' in x or 'full-layout' in x))
        cont = cont_tag.get_text().strip() if cont_tag else "내용 확인 불가"
    except:
        cont = "내용 확인 불가"
    print('3.내용:', cont)
    cont2.append(cont)      

    # 4. 작성일자 찾기 (사용자님이 쓰셨던 'subtexts' 단어 추적)
    try:
        date_tag = b.find(class_=lambda x: x and 'subtexts' in x)
        date = date_tag.get_text().strip() if date_tag else "날짜 확인 불가"
    except:
        date = "날짜 확인 불가"
    print('4.작성일자:' , date)
    date2.append(date)      
    
    # 5. 블로그닉네임 찾기 (사용자님이 쓰셨던 'profile-info-title' 단어 추적)
    try:
        name_tag = b.find(class_=lambda x: x and 'profile-info-title' in x)
        name = name_tag.get_text().strip() if name_tag else "닉네임 확인 불가"
    except:
        name = "닉네임 확인 불가"
    print('5.블로그닉네임:' , name)
    name2.append(name)      

    print("=" * 50) 

    if no >= collect_cnt :       
        break       

    no += 1

print("\n요청하신 작업이 모두 완료되었습니다")


# Step 10. 수집된 데이터를 xlsx와 csv 형태로 저장하기
import pandas as pd		#117~123까지 표 만드는 기능 (pandas 사용) 

# 표 만들어줘. 참고로 데이터프레임은 **빈칸** 있으면 안됨. 0으로 채우거나 아무 글자 집어넣어야함.
# 다만! pd.DataFrame() 이건 빈칸 있어도 그대로 해달라는 뜻임. 
df = pd.DataFrame()		
df['번호']=no2
df['제목']=pd.Series(title2)
df['내용']=pd.Series(cont2)
df['작성일자']=pd.Series(date2)
df['블로그닉네임']=pd.Series(name2)

# xls 형태로 저장하기
df.to_excel(fx_name, index=False, engine='openpyxl')		#표이름. 표이름을 엑셀로 저장해달라는 뜻
#engine='openpyxl' 이건 안 써도 됨. 웬만하면 자동으로 되어있음.

# csv 형태로 저장하기
df.to_csv(fc_name, index=False, encoding="utf-8-sig")
print('요청하신 데이터 수집 작업이 정상적으로 완료되었습니다')

 이 크롤러는 NAVER 블로그 수집용 웹크롤러입니다.
👉 수집된 데이터는 [C:\py_temp\추출 후 파일 저장\] 폴더에 자동 저장됩니다.
15 건을 수집하기 위해 페이지를 조회합니다.

요청하신 작업이 모두 완료되었습니다
요청하신 데이터 수집 작업이 정상적으로 완료되었습니다
